# End-to-End Multimodal Relational Deep Learning: Step-by-Step
Welcome! This notebook will walk you through the entire pipeline step-by-step so you can see exactly how the raw database is transformed into a graph, and how the model learns from it.

In [ ]:
import torch
import pandas as pd
import yaml
from rdl_e2e.data import RelationalDatabase, build_hetero_graph
from rdl_e2e.encoders import Tokenizer
from rdl_e2e.model import RDLModel
from rdl_e2e.train import train_rdl, TrainConfig

### Step 1: Loading the Relational Database
First, we load the raw relational database. Under the hood, this is just a collection of Pandas DataFrames connected by foreign keys.

In [ ]:
# We load the Formula 1 dataset (rel-f1) and the driver-dnf task
db = RelationalDatabase.from_relbench('rel-f1', 'driver-dnf')

# Let's look at the available tables:
print('Tables in the database:', list(db.tables.keys()))

# Let's peek at the drivers table!
display(db.tables['drivers'].head(3))

### Step 2: Converting the Database into a Heterogeneous Graph
A GNN needs a graph, not tables. The `build_hetero_graph` function converts every row into a node, and every foreign key into an edge.

In [ ]:
built_graph = build_hetero_graph(db)
graph = built_graph.graph

print(graph)
print("\nLook at the graph structure above! It shows the number of nodes per table, and the edges connecting them.")

### Step 3: Handling Text with Tokenizers
Before we can pass this graph to a model, we need to convert text columns into tokens (numbers).

In [ ]:
tokenizers = {}
for node_type, col_types in built_graph.column_types.items():
    if "text" in col_types.values():
        text_cols = [c for c, t in col_types.items() if t == "text"]
        corpus = [str(v) for c in text_cols for v in built_graph.raw_features[node_type][c]]
        # We initialize a tokenizer for these text columns
        tokenizers[node_type] = Tokenizer("distilbert-base-uncased", corpus, max_len=32)
        print(f"Created tokenizer for table: {node_type}")

### Step 4: Initializing the GNN Model
Now we create the `RDLModel`. This model automatically builds Graph Neural Network layers to match the exact schema of our database.

In [ ]:
node_types = list(graph.node_types)
edge_types = list(graph.edge_types)

model = RDLModel(
    node_types=node_types, 
    edge_types=edge_types, 
    entity_table=db.task.entity_table,
    node_column_types=built_graph.column_types, 
    tokenizers=tokenizers, 
    raw_features=built_graph.raw_features,
    text_model_name="distilbert-base-uncased", 
    trainable_text=False, # Set to True for End-to-End mode!
    hid=64, 
    n_gnn_layers=2, 
    out_dim=1
)

print(model)